# 11.6 Feature Engineering for Time Series

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.06_feature-engineering-for-time-series.ipynb)

Companion code for Section 11.6 of the book. Run the cells in order from top to bottom.

**Example 11.6.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings; warnings.filterwarnings('ignore')

# ── Generate realistic retail sales time series ───────────────────
np.random.seed(42)
n = 365 * 2   # 2 years daily data
dates = pd.date_range('2022-01-01', periods=n, freq='D')
t = np.arange(n)

# Compose realistic sales signal
trend    = 1000 + 0.5*t
weekly   = 200*np.sin(2*np.pi*t/7 - np.pi/2)   # Weekend peak
annual   = 300*np.sin(2*np.pi*t/365 - np.pi)    # Summer dip
holidays = np.zeros(n)
# Christmas spike
for yr in range(2022, 2024):
    idx = (dates.year==yr) & (dates.month==12) & (dates.day.isin(range(20,32)))
    holidays[idx] = 500
noise    = np.random.normal(0, 50, n)

sales    = trend + weekly + annual + holidays + noise
df = pd.DataFrame({'date': dates, 'sales': sales})
df = df.set_index('date')

print(f'Dataset: {len(df)} days from {df.index.min().date()} to {df.index.max().date()}')
print(f'Sales: mean={df.sales.mean():.0f}  std={df.sales.std():.0f}  range=[{df.sales.min():.0f},{df.sales.max():.0f}]')

# ── Complete Feature Engineering Pipeline ─────────────────────────
def create_ts_features(df, target_col='sales', lags=None, roll_windows=None):
    '''Create comprehensive time series features from a DataFrame.'''
    df = df.copy()
    y  = df[target_col]

    # 1. Calendar features
    df['day_of_week']    = df.index.dayofweek          # 0=Monday, 6=Sunday
    df['day_of_month']   = df.index.day
    df['day_of_year']    = df.index.dayofyear
    df['week_of_year']   = df.index.isocalendar().week.astype(int)
    df['month']          = df.index.month
    df['quarter']        = df.index.quarter
    df['year']           = df.index.year
    df['is_weekend']     = (df.index.dayofweek >= 5).astype(int)
    df['is_month_start'] = df.index.is_month_start.astype(int)
    df['is_month_end']   = df.index.is_month_end.astype(int)
    df['is_quarter_end'] = df.index.is_quarter_end.astype(int)

    # 2. Cyclical encoding (preserves periodicity)
    df['dow_sin']   = np.sin(2*np.pi*df['day_of_week']/7)
    df['dow_cos']   = np.cos(2*np.pi*df['day_of_week']/7)
    df['month_sin'] = np.sin(2*np.pi*df['month']/12)
    df['month_cos'] = np.cos(2*np.pi*df['month']/12)
    df['doy_sin']   = np.sin(2*np.pi*df['day_of_year']/365.25)
    df['doy_cos']   = np.cos(2*np.pi*df['day_of_year']/365.25)

    # 3. Lag features
    if lags is None:
        lags = [1, 2, 3, 7, 14, 21, 28, 30, 60, 90, 180, 365]
    for lag in lags:
        df[f'lag_{lag}'] = y.shift(lag)

    # 4. Rolling statistics
    if roll_windows is None:
        roll_windows = [7, 14, 21, 28, 60, 90]
    for w in roll_windows:
        df[f'roll_mean_{w}']   = y.shift(1).rolling(w).mean()
        df[f'roll_std_{w}']    = y.shift(1).rolling(w).std()
        df[f'roll_min_{w}']    = y.shift(1).rolling(w).min()
        df[f'roll_max_{w}']    = y.shift(1).rolling(w).max()
        df[f'roll_median_{w}'] = y.shift(1).rolling(w).median()

    # 5. Differencing features
    df['diff_1']     = y.diff(1)
    df['diff_7']     = y.diff(7)
    df['diff_365']   = y.diff(365)

    # 6. Exponential weighted features
    df['ewm_7']  = y.shift(1).ewm(span=7).mean()
    df['ewm_28'] = y.shift(1).ewm(span=28).mean()

    # 7. Date-based ordinal features
    df['days_from_start'] = (df.index - df.index.min()).days

    return df


# Apply feature engineering
df_feat = create_ts_features(df)
df_feat = df_feat.dropna()  # Drop rows with NaN from lag/rolling

n_features = len([c for c in df_feat.columns if c != 'sales'])
print(f'\nFeature engineering: 1 column → {n_features} features')
print(f'Rows after dropping NaN: {len(df_feat)} (removed {len(df)-len(df_feat)} for warmup)')

# Feature categories
feature_cats = {
    'Calendar':  [c for c in df_feat.columns if c in
['day_of_week','month','quarter','year','is_weekend','is_month_start','is_month_end','is_quarter_end','day_of_year']],
    'Cyclical':  [c for c in df_feat.columns if 'sin' in c or 'cos' in c],
    'Lag':       [c for c in df_feat.columns if c.startswith('lag_')],
    'Rolling':   [c for c in df_feat.columns if c.startswith('roll_')],
    'Diff':      [c for c in df_feat.columns if c.startswith('diff_')],
    'EWM':       [c for c in df_feat.columns if c.startswith('ewm_')],
}
print('\nFeature breakdown:')
for cat, feats in feature_cats.items():
    print(f'  {cat:<10}: {len(feats)} features  (e.g. {feats[:3]})')

# ── Train ML model with engineered features ───────────────────────
feature_cols = [c for c in df_feat.columns if c != 'sales']
X = df_feat[feature_cols]
y = df_feat['sales']

# Walk-forward split: last 60 days as test
cutoff = df_feat.index[-60]
X_tr = X[df_feat.index < cutoff]
X_te = X[df_feat.index >= cutoff]
y_tr = y[df_feat.index < cutoff]
y_te = y[df_feat.index >= cutoff]

model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                    max_depth=4, random_state=42)
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)

mae  = mean_absolute_error(y_te, y_pred)
mape = np.mean(np.abs((y_te - y_pred) / y_te)) * 100
rmse = np.sqrt(mean_squared_error(y_te, y_pred))
print(f'\nML Forecast (60-day test):')
print(f'  MAE:  {mae:.2f}')
print(f'  MAPE: {mape:.2f}%')
print(f'  RMSE: {rmse:.2f}')

# ── Feature importance ────────────────────────────────────────────
fi = pd.Series(model.feature_importances_, index=feature_cols)
fi = fi.sort_values(ascending=False)
print('\nTop 10 Important Features:')
print(fi.head(10).round(4))